# Predicting Student Health Risk - EDA

This notebook is the public exploratory pass for Kaggle Playground Series S6E7. The goal is not to over-model yet; it is to make the dataset legible, identify the target imbalance, inspect missingness, and turn those observations into a clean modeling checklist.

**Working questions**

- What is the target and how imbalanced is it?
- Which fields are numeric versus categorical?
- Is missingness large enough to become a signal rather than simple cleanup?
- Which features actually carry signal, measured rather than assumed?
- Are there exact train/test duplicate rows worth exploiting?
- Do train and test distributions look broadly aligned, including multivariate shifts a single-feature histogram would miss?
- What should the first baseline optimize and diagnose?

## 1. Setup

The data path resolver supports both Kaggle execution and local development. On Kaggle, competition data can mount under different folder names, so the notebook scans `/kaggle/input` when the common paths are absent.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42

candidate_dirs = [
    Path('/kaggle/input/playground-series-s6e7'),
    Path('/kaggle/input/predicting-student-health-risk'),
    Path('../data'),
    Path('data'),
]
DATA_DIR = next(
    (path for path in candidate_dirs if (path / 'train.csv').exists()),
    None,
)
if DATA_DIR is None and Path('/kaggle/input').exists():
    for train_path in Path('/kaggle/input').glob('**/train.csv'):
        parent = train_path.parent
        if (parent / 'test.csv').exists() and (parent / 'sample_submission.csv').exists():
            DATA_DIR = parent
            break
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train/test/sample_submission CSV files.')

sns.set_theme(style='whitegrid', palette='viridis')
pd.set_option('display.max_columns', 120)
print(f'Using data directory: {DATA_DIR}')

## 2. Load The Competition Files

The competition follows the standard Playground shape: `train.csv`, `test.csv`, and `sample_submission.csv`. The only column present in train but absent from test is treated as the target.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

target_candidates = [col for col in train.columns if col not in test.columns]
target = target_candidates[0]
id_col = sample_submission.columns[0]

summary = pd.DataFrame(
    {
        'rows': [len(train), len(test), len(sample_submission)],
        'columns': [train.shape[1], test.shape[1], sample_submission.shape[1]],
    },
    index=['train', 'test', 'sample_submission'],
)
display(summary)
print(f'Target column: {target}')
display(train.head())
display(sample_submission.head())

## 3. Target Balance

`health_condition` is highly imbalanced. Plain accuracy will be dominated by the majority `at-risk` class, so every serious model should also report class-balanced metrics such as macro F1, balanced accuracy, and per-class recall.

In [ ]:
target_counts = train[target].value_counts(dropna=False)
target_summary = pd.DataFrame(
    {
        'count': target_counts,
        'share': target_counts / len(train),
    }
)
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    x=target_summary.index,
    y=target_summary['share'],
    hue=target_summary.index,
    palette='viridis',
    legend=False,
    ax=ax,
)
ax.set_title('Target class share')
ax.set_xlabel('health_condition')
ax.set_ylabel('Share of training rows')
ax.bar_label(ax.containers[0], fmt='%.1%%')
plt.tight_layout()
plt.show()

## 4. Feature Groups

The dataset mixes continuous lifestyle/health measurements with low-cardinality categorical fields. This is a good fit for tree-based tabular models, especially CatBoost/LightGBM after the first sklearn baseline.

In [ ]:
feature_cols = [col for col in test.columns if col in train.columns and col != id_col]
num_cols = train[feature_cols].select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in feature_cols if col not in num_cols]

print(f'ID column: {id_col}')
print(f'Numeric features ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

display(train[num_cols].describe().T)
cat_summary = pd.DataFrame(
    {
        'train_unique': train[cat_cols].nunique(dropna=False),
        'test_unique': test[cat_cols].nunique(dropna=False),
        'train_missing': train[cat_cols].isna().sum(),
        'test_missing': test[cat_cols].isna().sum(),
    }
)
display(cat_summary)

## 5. Missingness Profile

There are many missing values. For the first baseline, this should be handled explicitly with imputation plus missingness indicators. Later experiments should test whether missingness patterns correlate with `health_condition`.

In [ ]:
missing = pd.DataFrame(
    {
        'train_missing': train[feature_cols].isna().sum(),
        'train_missing_share': train[feature_cols].isna().mean(),
        'test_missing': test[feature_cols].isna().sum(),
        'test_missing_share': test[feature_cols].isna().mean(),
    }
).sort_values('train_missing_share', ascending=False)

display(missing[missing[['train_missing', 'test_missing']].sum(axis=1) > 0])

row_missing = pd.DataFrame(
    {
        'train_missing_features': train[feature_cols].isna().sum(axis=1),
        'target': train[target],
    }
)
display(row_missing.groupby('target')['train_missing_features'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=missing.reset_index().rename(columns={'index': 'feature'}),
    x='train_missing_share',
    y='feature',
    color=sns.color_palette('viridis', 5)[2],
    ax=ax,
)
ax.set_title('Missing-value share by feature')
ax.set_xlabel('Train missing share')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 6. Feature Relevance: Mutual Information

Mutual information (MI) measures how much each feature tells you about the class on
its own. It is a quick relevance ranking, not the last word — a high-MI feature can
still add little once correlated features are already in a model — but it confirms
which columns deserve attention before building anything.


In [ ]:
mi_frame = train[feature_cols].copy()
for col in cat_cols:
    mi_frame[col] = mi_frame[col].astype('category').cat.codes  # -1 for missing
for col in num_cols:
    mi_frame[col] = mi_frame[col].fillna(mi_frame[col].median())

discrete_mask = [col in cat_cols for col in feature_cols]
mutual_info = pd.Series(
    mutual_info_classif(mi_frame[feature_cols], train[target], discrete_features=discrete_mask, random_state=SEED),
    index=feature_cols,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
sns.barplot(
    x=mutual_info.values,
    y=mutual_info.index,
    hue=mutual_info.index,
    palette='viridis',
    legend=False,
    ax=ax,
)
ax.set_title('Feature relevance: mutual information with health_condition')
ax.set_xlabel('Mutual information')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

display(mutual_info.round(4).rename('mutual_information').to_frame())


`sleep_duration` and `stress_level` lead by a clear margin, `physical_activity_level`
and `bmi` form a second tier, and `diet_type`/`gender` carry almost no independent
signal. This lines up with the target-balance and category crosstabs above, and gives
the first baseline a concrete priority order instead of guessing from eyeballed plots.


## 7. Exact Train/Test Duplicate Check

Playground Series data is generated from an underlying source table with noise added.
In some competitions in this series, test rows land byte-identical to a train row across
every feature, which means the label is not predicted, it is known outright. This checks
directly whether that lever exists here, before any modeling assumes it does or doesn't.


In [ ]:
combined_features = pd.concat([train[feature_cols], test[feature_cols]], axis=0, ignore_index=True)
combined_filled = combined_features.fillna(-999999)  # sentinel so NaN participates in the group key
group_sizes = combined_filled.groupby(feature_cols).size()

print(f'Combined train+test rows: {len(combined_filled):,}')
print(f'Distinct exact-feature combinations: {len(group_sizes):,}')
print(f'Largest group of identical rows: {group_sizes.max()}')
print(f'Rows sharing an exact duplicate with >=1 other row: {int(group_sizes[group_sizes > 1].sum()):,}')


**Zero exact duplicates**: every one of the combined rows is a distinct combination of
features. Seven largely-independent continuous numeric columns make this dataset
high-dimensional enough that rows never collide, even under this competition's synthetic
generation. The exact-duplicate-row lever that pays off in some other Playground Series
entries does not apply here — worth ruling out explicitly rather than assuming it either
way.


## 8. Train/Test Drift: Quick Screen + Adversarial Validation

This is a lightweight first check. Numeric features compare mean and standard deviation shifts. Categorical features compare the most common train/test values. Larger drift deserves fold-aware validation and cautious leaderboard interpretation.

In [ ]:
num_drift = pd.DataFrame(index=num_cols)
num_drift['train_mean'] = train[num_cols].mean()
num_drift['test_mean'] = test[num_cols].mean()
num_drift['mean_delta'] = num_drift['test_mean'] - num_drift['train_mean']
num_drift['train_std'] = train[num_cols].std()
num_drift['test_std'] = test[num_cols].std()
num_drift['std_delta'] = num_drift['test_std'] - num_drift['train_std']
display(num_drift.sort_values('mean_delta', key=lambda s: s.abs(), ascending=False))

cat_rows = []
for col in cat_cols:
    train_top = train[col].value_counts(dropna=False, normalize=True).head(3)
    test_top = test[col].value_counts(dropna=False, normalize=True).head(3)
    cat_rows.append(
        {
            'feature': col,
            'train_top_values': ', '.join(f'{idx}: {val:.1%}' for idx, val in train_top.items()),
            'test_top_values': ', '.join(f'{idx}: {val:.1%}' for idx, val in test_top.items()),
        }
    )
display(pd.DataFrame(cat_rows))

Histograms only catch shifts you can see one feature at a time. A collective shift —
several features each nudged a little — can hide from them. The direct check: train a
classifier to tell train rows from test rows. If it cannot beat chance (ROC-AUC 0.5), the
two sets are interchangeable and a plain split on train is safe to trust.


In [ ]:
adv_features = pd.concat([train[feature_cols], test[feature_cols]], axis=0, ignore_index=True).copy()
for col in cat_cols:
    adv_features[col] = adv_features[col].astype('category').cat.codes
adv_target = np.r_[np.ones(len(train)), np.zeros(len(test))].astype(int)

adv_train_X, adv_val_X, adv_train_y, adv_val_y = train_test_split(
    adv_features, adv_target, test_size=0.2, stratify=adv_target, random_state=SEED
)
adv_model = HistGradientBoostingClassifier(random_state=SEED, max_iter=200, learning_rate=0.05)
adv_model.fit(adv_train_X, adv_train_y)
adv_auc = roc_auc_score(adv_val_y, adv_model.predict_proba(adv_val_X)[:, 1])
print(f'Adversarial train-vs-test ROC-AUC = {adv_auc:.4f}  (0.50 = indistinguishable)')

adv_val_sample = adv_val_X.sample(min(50_000, len(adv_val_X)), random_state=SEED)
adv_val_sample_y = adv_val_y[adv_val_X.index.get_indexer(adv_val_sample.index)]
perm_result = permutation_importance(
    adv_model, adv_val_sample, adv_val_sample_y,
    n_repeats=5, random_state=SEED, scoring='roc_auc', n_jobs=-1,
)
adv_importance = pd.Series(perm_result.importances_mean, index=feature_cols).sort_values(ascending=False)
display(adv_importance.round(4).rename('permutation_importance_auc').to_frame().head(6))


Train and test are distinguishable above chance (ROC-AUC `~0.65`), so a mild
*multivariate* shift is present that the per-feature histograms above miss on their own —
concentrated in `water_intake`, `physical_activity_level`, `calorie_expenditure`, and
`bmi`. That is nowhere near the `~0.8+` AUC that would justify importance-weighting
training rows or distrusting a plain stratified split; the honest read is "mild, and safe
to leave alone," not "no drift at all." Worth keeping this check in the template — the
shift would have stayed invisible to the histogram screen above.


## 9. Modeling Implications

**Immediate choices**

- Use stratified cross-validation because the target is very imbalanced.
- Report macro F1, balanced accuracy, and per-class recall alongside any official metric.
- Add missing-value indicators to the baseline preprocessing.
- Prefer tree-based models for the first serious pass: HistGradientBoosting, LightGBM, XGBoost, and CatBoost.
- Validate feature families around sleep/stress/activity interactions instead of adding a large untracked feature pile.
- Prioritize `sleep_duration` and `stress_level` first (§6); `diet_type`/`gender` carry almost no independent signal on their own.
- Do not spend effort hunting for exact-duplicate train/test rows (§7) — that lever is confirmed empty for this dataset.
- Train/test drift is mild (§8, adversarial ROC-AUC `~0.65`) and concentrated in `water_intake`/`calorie_expenditure`/`bmi` — a plain stratified split is safe; no importance-weighting needed.

**Next notebook:** `02_baseline_modeling.ipynb` builds the fold-safe baseline and every subsequent experiment. **Update, project closed:** the modeling phase that notebook documents is now finished — `lgbm_xgb_domain_ensemble` (v8) is the final champion at public `0.94959`, after five further levers (v19-v23) all failed to clear the promotion gate. See `docs/7_leaderboard_improvement_insights.md` for the full ledger and the external-research writeup on why that score is a documented ceiling for this dataset, not a gap in this project's modeling.